# Market Data Fetching from Yahoo Finance
Real-time market data retrieval

In [ ]:
from utils.market_data_utils import fetch_spot_price, fetch_volatility, fetch_risk_free_rate
from utils.common import timer
import pandas as pd

## Fetch Single Stock Data

In [ ]:
@timer
def get_stock_data(ticker):
    return fetch_spot_price(ticker)

aapl_data = get_stock_data('AAPL')

print(f"Ticker: {aapl_data['ticker']}")
print(f"Spot Price: ${aapl_data['spot']:.2f}")
print(f"Previous Close: ${aapl_data['prev_close']:.2f}")
print(f"Change: ${aapl_data['change']:.2f} ({aapl_data['change_pct']:.2f}%)")
print(f"Volume: {aapl_data['volume']:,}")

## Batch Fetch Multiple Tickers

In [ ]:
tickers = ['AAPL', 'TSLA', 'SPY', 'NVDA', 'MSFT']

@timer
def batch_fetch(tickers):
    return [fetch_spot_price(t) for t in tickers]

batch_data = batch_fetch(tickers)

# Create DataFrame
df = pd.DataFrame(batch_data)
df = df[['ticker', 'spot', 'change_pct', 'volume']]
df['volume'] = df['volume'].apply(lambda x: f"{x/1e6:.1f}M")
df['change_pct'] = df['change_pct'].apply(lambda x: f"{x:.2f}%")
df['spot'] = df['spot'].apply(lambda x: f"${x:.2f}")

print(df.to_string(index=False))

## Fetch Volatility Data

In [ ]:
vol_data = fetch_volatility('TSLA', lookback_days=30)

print(f"Ticker: {vol_data['ticker']}")
print(f"Historical Vol (30d): {vol_data['historical_vol']:.1%}")
print(f"Implied Vol (mock): {vol_data['implied_vol']:.1%}")
print(f"Vol Spread: {vol_data['vol_spread']:.1%}")

## Complete Option Pricing Parameters

In [ ]:
# Gather all parameters needed for option pricing
ticker = 'SPY'

# Spot price
spot_data = fetch_spot_price(ticker)
spot = spot_data['spot']

# Volatility
vol_data = fetch_volatility(ticker, 30)
volatility = vol_data['implied_vol']

# Risk-free rate
risk_free_rate = fetch_risk_free_rate()

print("Option Pricing Parameters:")
print("=" * 30)
print(f"Underlying: {ticker}")
print(f"Spot Price: ${spot:.2f}")
print(f"Volatility: {volatility:.1%}")
print(f"Risk-Free Rate: {risk_free_rate:.2%}")
print(f"\nReady for pricing!")

## Error Handling Demo

In [ ]:
# Try invalid ticker
try:
    invalid_data = fetch_spot_price('INVALID123')
except Exception as e:
    print(f"Error fetching INVALID123: {e}")

# Successful fetch
valid_data = fetch_spot_price('GOOGL')
print(f"\nSuccessfully fetched GOOGL: ${valid_data['spot']:.2f}")